# Real-Source Korean FX Deposit Rate Pipeline

This notebook builds a **fully reproducible** Python pipeline for collecting **real** (not estimated) foreign currency deposit rates reported by major Korean banks for 2004–2019.

## Data Source Strategy (Priority Order)

1. **ECOS (Bank of Korea Economic Statistics System)** — official bank-reported rates published by the central bank
2. **FSS FISIS (Financial Supervisory Service)** — supervisory disclosure data
3. **Wayback Machine** — archived snapshots of official bank rate pages
4. **Current official bank pages** — live rate tables
5. **Manual official disclosures** — locally stored official rate tables with provenance

## Scope

| Dimension | Specification |
|-----------|--------------|
| Banks | Shinhan, Woori, KEB/Hana, IBK, KB Kookmin, NH Nonghyup, SC Korea |
| Currencies | USD, JPY, CNY (required) · EUR, GBP (optional) |
| Period | 2004–2019 |
| Frequency | Annual (configurable to quarterly/monthly) |

## Strict Rules

- **NO** synthetic data, LIBOR proxies, benchmark+spread approximations
- **NO** interpolation, smoothing, or fabricated values
- Every kept observation **MUST** trace to a real source URL

## Deliverables

| File | Contents |
|------|----------|
| `fx_deposit_rates_raw.csv` | All scraped observations |
| `fx_deposit_rates_clean.csv` | Deduplicated, priority-ranked |
| `fx_deposit_rates_panel.xlsx` | Sheets: raw, clean, strict_panel, extended_panel, source_log, missing_report |

In [ ]:
%pip install -q pandas requests beautifulsoup4 lxml openpyxl xlsxwriter matplotlib python-dateutil

from __future__ import annotations

import io
import json
import re
import time
import warnings
from itertools import combinations
from pathlib import Path
from urllib.parse import quote, urljoin, urlparse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.unicode_minus"] = False
pd.options.display.max_columns = 60
pd.options.display.width = 160

print("Libraries installed and imported.")

## 2. Configuration: Paths, Bank Registry, ECOS API, Currencies, and Patterns

- Output paths and reproducible run config
- **ECOS API** configuration with known stat codes for FX deposit rates
- Bank source registry with official domains, landing URLs, and known rate-page paths
- Currency, product, maturity, and resident pattern dictionaries

In [ ]:
ROOT = Path.cwd()
OUTPUT_RAW_CSV = ROOT / "fx_deposit_rates_raw.csv"
OUTPUT_CLEAN_CSV = ROOT / "fx_deposit_rates_clean.csv"
OUTPUT_PANEL_XLSX = ROOT / "fx_deposit_rates_panel.xlsx"
OUTPUT_DROPPED_CSV = ROOT / "fx_deposit_dropped_duplicates.csv"
LOG_FILE = ROOT / "fx_deposit_pipeline.log"
FIG_DIR = ROOT / "fx_rate_figures"
MANUAL_DIR = ROOT / "manual_sources"
MANIFEST_FILE = MANUAL_DIR / "manual_source_manifest.csv"

FIG_DIR.mkdir(exist_ok=True)
MANUAL_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# ECOS (Bank of Korea) API configuration
# ---------------------------------------------------------------------------
ECOS_CONFIG = {
    "api_key": "LZ5FB0NNZDPJFXE6ZVTL",
    "base_url": "https://ecos.bok.or.kr/api",
    "stat_codes": [
        ("722Y001", "외화예금/대출금리 (Foreign Currency Deposit/Loan Rates)"),
        ("121Y002", "예금은행 외화대출금리 (Deposit Bank Foreign Loan Rates)"),
        ("121Y015", "예금은행 예금금리 (Deposit Bank Deposit Rates)"),
        ("121Y003", "예금은행 대출금리 (Deposit Bank Loan Rates)"),
        ("038Y201", "통화금융기관 외화예수금 (Monetary Institutions FX Deposits)"),
        ("064Y001", "외화예금 (Foreign Currency Deposits)"),
    ],
}

# ---------------------------------------------------------------------------
# Run configuration
# ---------------------------------------------------------------------------
RUN_CONFIG = {
    "start_date": "2004-01-01",
    "end_date": "2019-12-31",
    "frequency": "A",  # Supported: A, Q, M
    "request_timeout": 25,
    "pause_seconds": 0.35,
    "max_current_links": 12,
    "max_wayback_pages_per_date": 5,
    "max_tables_per_page": 20,
    "max_cdx_snapshots": 30,
    "required_currencies": ["USD", "JPY", "CNY"],
    "optional_currencies": ["EUR", "GBP"],
}

# ---------------------------------------------------------------------------
# Bank source registry — domains, landing URLs, known rate page paths
# ---------------------------------------------------------------------------
BANK_SOURCE_REGISTRY = [
    {
        "bank": "Shinhan Bank",
        "bank_group": "commercial_bank",
        "aliases": ["shinhan", "shinhan bank", "신한", "신한은행"],
        "domains": ["www.shinhan.com", "bank.shinhan.com"],
        "landing_urls": [
            "https://www.shinhan.com/",
            "https://bank.shinhan.com/",
        ],
        "known_rate_paths": [
            "https://www.shinhan.com/hpe/customer/CS08/CS08008RP01.xml",
            "https://bank.shinhan.com/index.jsp#050504000000",
        ],
    },
    {
        "bank": "Woori Bank",
        "bank_group": "commercial_bank",
        "aliases": ["woori", "woori bank", "우리", "우리은행"],
        "domains": ["www.wooribank.com", "spot.wooribank.com", "spib.wooribank.com"],
        "landing_urls": [
            "https://www.wooribank.com/",
            "https://spot.wooribank.com/",
        ],
        "known_rate_paths": [
            "https://spot.wooribank.com/pot/Dream?withyou=FXIEN0054",
            "https://spib.wooribank.com/pib/Dream?withyou=BISMP010",
        ],
    },
    {
        "bank": "KEB/Hana Bank",
        "bank_group": "commercial_bank",
        "aliases": ["hana", "hana bank", "keb", "keb hana", "하나", "하나은행", "외환", "외환은행"],
        "domains": ["www.kebhana.com", "www.hanabank.com"],
        "landing_urls": [
            "https://www.kebhana.com/",
            "https://www.hanabank.com/",
        ],
        "known_rate_paths": [
            "https://www.kebhana.com/cont/mall/mall08/index.jsp",
            "https://www.kebhana.com/cont/mall/mall08/mall0805/mall080501/index.jsp",
        ],
    },
    {
        "bank": "Industrial Bank of Korea (IBK)",
        "bank_group": "policy_bank",
        "aliases": ["ibk", "industrial bank of korea", "기업은행", "ibk 기업은행"],
        "domains": ["www.ibk.co.kr", "mybank.ibk.co.kr"],
        "landing_urls": [
            "https://www.ibk.co.kr/",
            "https://mybank.ibk.co.kr/",
        ],
        "known_rate_paths": [
            "https://mybank.ibk.co.kr/uib/jsp/index.jsp",
            "https://www.ibk.co.kr/lang/en/rate/depositsInterest.jsp",
        ],
    },
    {
        "bank": "KB Kookmin Bank",
        "bank_group": "commercial_bank",
        "aliases": ["kb", "kookmin", "kookmin bank", "국민", "국민은행", "kb국민은행"],
        "domains": ["www.kbstar.com", "obank.kbstar.com"],
        "landing_urls": [
            "https://www.kbstar.com/",
            "https://obank.kbstar.com/",
        ],
        "known_rate_paths": [
            "https://obank.kbstar.com/quics?page=C101407",
        ],
    },
    {
        "bank": "NH Nonghyup Bank",
        "bank_group": "commercial_bank",
        "aliases": ["nh", "nonghyup", "nonghyup bank", "농협", "농협은행", "nh농협은행"],
        "domains": ["banking.nonghyup.com", "www.nonghyup.com"],
        "landing_urls": [
            "https://banking.nonghyup.com/",
            "https://www.nonghyup.com/",
        ],
        "known_rate_paths": [
            "https://banking.nonghyup.com/nhbank.html",
        ],
    },
    {
        "bank": "Standard Chartered Korea",
        "bank_group": "foreign_bank",
        "aliases": ["standard chartered", "sc", "sc bank", "제일은행", "sc제일은행", "standard chartered korea"],
        "domains": ["www.standardchartered.co.kr"],
        "landing_urls": [
            "https://www.standardchartered.co.kr/",
        ],
        "known_rate_paths": [
            "https://www.standardchartered.co.kr/np/kr/pl/etb/rates.jsp",
        ],
    },
]

# ---------------------------------------------------------------------------
# Pattern dictionaries for label matching
# ---------------------------------------------------------------------------
CURRENCY_PATTERNS = {
    "USD": [r"\busd\b", r"u\.s\.\s*dollar", r"us\s*dollar", r"미국\s*달러", r"(?<!\w)달러(?!\w)", r"usdollar", r"미달러"],
    "JPY": [r"\bjpy\b", r"\byen\b", r"일본\s*엔", r"엔화", r"(?<![a-zA-Z])엔(?![a-zA-Z가-z])"],
    "CNY": [r"\bcny\b", r"\brmb\b", r"renminbi", r"\byuan\b", r"위안", r"위안화", r"인민폐", r"인민币", r"중국\s*위안"],
    "EUR": [r"\beur\b", r"\beuro\b", r"유로"],
    "GBP": [r"\bgbp\b", r"\bpound\b", r"\bsterling\b", r"영국\s*파운드", r"파운드"],
}

PRODUCT_PATTERNS = {
    "term deposit": [r"정기예금", r"time\s*deposit", r"term\s*deposit", r"fixed\s*deposit"],
    "ordinary deposit": [r"보통예금", r"ordinary\s*deposit", r"demand\s*deposit", r"입출금"],
    "savings deposit": [r"저축예금", r"savings?\s*deposit"],
    "other": [r"예금", r"deposit"],
}

MATURITY_PATTERNS = {
    "12M": [r"12\s*m", r"12개월", r"1\s*년", r"one\s*year", r"1\s*y(?:ear)?"],
    "6M": [r"6\s*m", r"6개월", r"six\s*month"],
    "3M": [r"3\s*m", r"3개월", r"three\s*month"],
    "1M": [r"1\s*m", r"1개월", r"one\s*month"],
    "ON": [r"overnight", r"수시", r"\bcall\b"],
}

RESIDENT_PATTERNS = {
    "resident": [r"거주자", r"\bresident\b"],
    "non_resident": [r"비거주자", r"non[-\s]?resident"],
}

SOURCE_PRIORITY = {"manual": 0, "ecos": 1, "fss": 2, "current": 3, "wayback": 4}
PRODUCT_PRIORITY = {"term deposit": 0, "ordinary deposit": 1, "savings deposit": 2, "other": 9, "unknown": 99}
MATURITY_PRIORITY = {"12M": 0, "6M": 1, "3M": 2, "1M": 3, "ON": 4, "unknown": 99}

REQUIRED_COLUMNS = [
    "bank", "bank_group", "date", "year", "quarter", "frequency",
    "currency", "product_type", "maturity", "resident_flag",
    "rate_percent", "source_url", "source_type",
]

EXTRA_COLUMNS = [
    "source_original_url", "snapshot_timestamp", "page_title",
    "raw_row_text", "raw_column_text",
]

SOURCE_EVENTS = []

# ECOS bank name mapping — maps ECOS item names to canonical bank names
ECOS_BANK_MAP = {
    "신한": "Shinhan Bank",
    "우리": "Woori Bank",
    "하나": "KEB/Hana Bank",
    "외환": "KEB/Hana Bank",
    "기업": "Industrial Bank of Korea (IBK)",
    "국민": "KB Kookmin Bank",
    "농협": "NH Nonghyup Bank",
    "sc": "Standard Chartered Korea",
    "제일": "Standard Chartered Korea",
    "한국스탠다드차타드": "Standard Chartered Korea",
}

if not MANIFEST_FILE.exists():
    pd.DataFrame(
        columns=["bank", "bank_group", "file_name", "source_url", "source_date", "notes"]
    ).to_csv(MANIFEST_FILE, index=False)

print(f"Working directory: {ROOT}")
print(f"Frequency: {RUN_CONFIG['frequency']}")
print(f"Banks configured: {len(BANK_SOURCE_REGISTRY)}")
print(f"ECOS stat codes to search: {len(ECOS_CONFIG['stat_codes'])}")

## 3. Core Infrastructure: HTTP, Logging, ECOS API, and Wayback Machine

This cell implements:

1. **Logging** — every request, parse, and quality event is captured for audit
2. **HTTP session** — cached, rate-limited requests with configurable timeout
3. **ECOS API client** — enumerate stat table items, pull time series, map to canonical bank/currency
4. **Wayback Machine** — CDX API for snapshot discovery + HTML fetch
5. **Link scoring** — heuristic ranking of candidate rate-page links

In [ ]:
HTTP_CACHE: dict = {}
WAYBACK_CACHE: dict = {}

session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9,ko;q=0.8",
    }
)

# ── Logging ──────────────────────────────────────────────────────────────────

def reset_logging_state() -> None:
    SOURCE_EVENTS.clear()
    LOG_FILE.write_text("", encoding="utf-8")


def log_event(
    level: str,
    bank: str,
    stage: str,
    message: str,
    source_url: str | None = None,
    source_type: str | None = None,
    target_date: str | None = None,
    extra: dict | None = None,
) -> None:
    event = {
        "event_time": pd.Timestamp.utcnow().isoformat(),
        "level": level,
        "bank": bank,
        "stage": stage,
        "message": message,
        "source_url": source_url,
        "source_type": source_type,
        "target_date": target_date,
        "extra": json.dumps(extra or {}, ensure_ascii=False),
    }
    SOURCE_EVENTS.append(event)
    with LOG_FILE.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(event, ensure_ascii=False) + "\n")


# ── Target dates ─────────────────────────────────────────────────────────────

def build_target_dates() -> list[pd.Timestamp]:
    start = pd.Timestamp(RUN_CONFIG["start_date"])
    end = pd.Timestamp(RUN_CONFIG["end_date"])
    freq = RUN_CONFIG["frequency"].upper()
    freq_map = {"M": "ME", "Q": "QE", "A": "YE"}
    return list(pd.date_range(start, end, freq=freq_map.get(freq, "YE")))


TARGET_DATES = build_target_dates()
FREQUENCY_LABEL = {"M": "monthly", "Q": "quarterly", "A": "annual"}[
    RUN_CONFIG["frequency"].upper()
]


def normalize_space(value: object) -> str:
    return re.sub(r"\s+", " ", str(value)).strip()


# ── HTTP helpers ─────────────────────────────────────────────────────────────

def strip_wayback_prefix(url: str) -> str:
    match = re.search(
        r"https?://web\.archive\.org/web/\d+(?:[a-z_]+)?/(https?://.+)", url
    )
    return match.group(1) if match else url


def same_bank_domain(url: str, domains: list[str]) -> bool:
    try:
        host = urlparse(strip_wayback_prefix(url)).netloc.lower()
    except Exception:
        return False
    return any(host.endswith(d.lower()) for d in domains)


def score_candidate_link(text: str, href: str) -> int:
    sample = f"{normalize_space(text).lower()} {normalize_space(href).lower()}"
    positive = ["금리", "예금", "외화", "deposit", "rate", "interest", "time", "term"]
    negative = ["loan", "대출", "card", "카드", "fund", "보험", "event", "login", "환율", "exchange"]
    score = sum(tok in sample for tok in positive) * 2
    score -= sum(tok in sample for tok in negative) * 3
    return score


def get_response_text(
    url: str, bank: str, stage: str, source_type: str
) -> dict:
    cache_key = (url, source_type)
    if cache_key in HTTP_CACHE:
        return HTTP_CACHE[cache_key]
    try:
        resp = session.get(
            url, timeout=RUN_CONFIG["request_timeout"], allow_redirects=True
        )
        if not resp.encoding:
            resp.encoding = resp.apparent_encoding or "utf-8"
        payload = {
            "ok": resp.ok,
            "status_code": resp.status_code,
            "final_url": resp.url,
            "text": resp.text,
            "content_type": resp.headers.get("Content-Type", ""),
        }
        if not resp.ok:
            log_event(
                "warning", bank, stage,
                f"HTTP {resp.status_code}", source_url=url, source_type=source_type
            )
    except Exception as exc:
        payload = {
            "ok": False,
            "status_code": None,
            "final_url": url,
            "text": "",
            "content_type": "",
            "error": str(exc),
        }
        log_event(
            "error", bank, stage, str(exc),
            source_url=url, source_type=source_type
        )
    HTTP_CACHE[cache_key] = payload
    time.sleep(RUN_CONFIG["pause_seconds"])
    return payload


def parse_html_bundle(html: str, base_url: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    title = normalize_space(soup.title.get_text(" ", strip=True)) if soup.title else ""
    page_text = normalize_space(soup.get_text(" ", strip=True))
    links: list[dict] = []
    for anchor in soup.select("a[href]"):
        href = anchor.get("href", "").strip()
        if not href:
            continue
        full_url = urljoin(base_url, href)
        links.append(
            {"text": normalize_space(anchor.get_text(" ", strip=True)), "url": full_url}
        )
    try:
        tables = pd.read_html(io.StringIO(html), displayed_only=False)
    except ValueError:
        tables = []
    return {"title": title, "page_text": page_text, "links": links, "tables": tables}


# ── ECOS API Client ─────────────────────────────────────────────────────────

def ecos_get_stat_items(stat_code: str) -> list[dict]:
    """Retrieve all item codes from an ECOS statistical table."""
    key = ECOS_CONFIG["api_key"]
    url = f"{ECOS_CONFIG['base_url']}/StatisticItemList/{key}/json/kr/1/1000/{stat_code}"
    try:
        resp = session.get(url, timeout=RUN_CONFIG["request_timeout"])
        data = resp.json()
        if "StatisticItemList" in data:
            items = data["StatisticItemList"].get("row", [])
            log_event(
                "info", "ECOS", "stat_items",
                f"[{stat_code}] {len(items)} items found",
                source_url=url, source_type="ecos",
            )
            return items
    except Exception as exc:
        log_event("error", "ECOS", "stat_items", str(exc), source_url=url, source_type="ecos")
    return []


def ecos_search_data(
    stat_code: str,
    item_code: str,
    start: str,
    end: str,
    cycle: str = "A",
) -> list[dict]:
    """Pull time-series data from ECOS for a single item code."""
    key = ECOS_CONFIG["api_key"]
    url = (
        f"{ECOS_CONFIG['base_url']}/StatisticSearch/{key}/json/kr/1/1000"
        f"/{stat_code}/{cycle}/{start}/{end}/{item_code}"
    )
    try:
        resp = session.get(url, timeout=RUN_CONFIG["request_timeout"])
        data = resp.json()
        if "StatisticSearch" in data:
            rows = data["StatisticSearch"].get("row", [])
            return rows
    except Exception as exc:
        log_event("error", "ECOS", "search_data", str(exc), source_url=url, source_type="ecos")
    return []


def ecos_item_to_bank(item_name: str) -> str | None:
    """Map an ECOS item name to a canonical bank name via ECOS_BANK_MAP."""
    item_lower = item_name.lower()
    for alias, canonical in ECOS_BANK_MAP.items():
        if alias.lower() in item_lower:
            return canonical
    return None


def ecos_item_to_currency(item_name: str) -> str | None:
    """Extract currency from an ECOS item name."""
    for canonical, patterns in CURRENCY_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, item_name, flags=re.IGNORECASE):
                return canonical
    return None


def ecos_item_to_maturity(item_name: str) -> str:
    for canonical, patterns in MATURITY_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, item_name, flags=re.IGNORECASE):
                return canonical
    return "unknown"


def ecos_item_to_product(item_name: str) -> str:
    for canonical, patterns in PRODUCT_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, item_name, flags=re.IGNORECASE):
                return canonical
    return "unknown"


def ecos_parse_time(time_str: str, cycle: str) -> pd.Timestamp | None:
    """Convert ECOS TIME field (e.g. '2010', '201003', '2010Q2') to Timestamp."""
    try:
        if cycle == "A":
            return pd.Timestamp(f"{time_str}-12-31")
        elif cycle == "Q":
            # format: 2010Q1
            m = re.match(r"(\d{4})Q(\d)", time_str)
            if m:
                year, q = int(m.group(1)), int(m.group(2))
                month = q * 3
                return pd.Timestamp(f"{year}-{month:02d}-01") + pd.offsets.MonthEnd(0)
        elif cycle == "M":
            return pd.Timestamp(f"{time_str[:4]}-{time_str[4:6]}-01") + pd.offsets.MonthEnd(0)
    except Exception:
        pass
    return None


def ecos_collect_fx_deposit_rates() -> list[dict]:
    """
    Enumerate ECOS stat tables and pull any FX deposit rate data.
    Returns records in the pipeline's standard schema.
    """
    all_records: list[dict] = []
    key = ECOS_CONFIG["api_key"]

    for stat_code, description in ECOS_CONFIG["stat_codes"]:
        log_event("info", "ECOS", "collect", f"Scanning {stat_code}: {description}", source_type="ecos")
        items = ecos_get_stat_items(stat_code)
        if not items:
            continue

        # Filter for FX-related items
        fx_keywords = [
            "외화", "달러", "USD", "JPY", "EUR", "CNY", "GBP", "RMB",
            "엔", "위안", "유로", "파운드", "dollar", "yen", "yuan",
        ]
        fx_items = [
            item for item in items
            if any(kw.lower() in item.get("ITEM_NAME", "").lower() for kw in fx_keywords)
        ]

        if not fx_items:
            log_event("info", "ECOS", "collect", f"[{stat_code}] No FX items among {len(items)} items", source_type="ecos")
            continue

        log_event("info", "ECOS", "collect", f"[{stat_code}] Found {len(fx_items)} FX items", source_type="ecos")

        for item in fx_items:
            item_code = item["ITEM_CODE"]
            item_name = item.get("ITEM_NAME", "")
            item_cycle = item.get("CYCLE", "A")

            currency = ecos_item_to_currency(item_name)
            bank = ecos_item_to_bank(item_name)
            product_type = ecos_item_to_product(item_name)
            maturity = ecos_item_to_maturity(item_name)

            if not currency:
                continue

            # Determine date range format by cycle
            cycles_to_try = [item_cycle] if item_cycle in ("A", "Q", "M") else ["A", "Q", "M"]
            for cycle in cycles_to_try:
                if cycle == "M":
                    start_dt, end_dt = "200401", "201912"
                elif cycle == "Q":
                    start_dt, end_dt = "2004Q1", "2019Q4"
                else:
                    start_dt, end_dt = "2004", "2019"

                data_rows = ecos_search_data(stat_code, item_code, start_dt, end_dt, cycle)
                if not data_rows:
                    continue

                freq_label = {"A": "annual", "Q": "quarterly", "M": "monthly"}.get(cycle, "annual")
                source_url = (
                    f"{ECOS_CONFIG['base_url']}/StatisticSearch/{key}/json/kr/1/1000"
                    f"/{stat_code}/{cycle}/{start_dt}/{end_dt}/{item_code}"
                )

                for row in data_rows:
                    val = row.get("DATA_VALUE", "")
                    if not val or val == "-":
                        continue
                    try:
                        rate = float(val.replace(",", ""))
                    except (ValueError, TypeError):
                        continue
                    if rate <= 0 or rate > 40:
                        continue

                    ts = ecos_parse_time(row.get("TIME", ""), cycle)
                    if ts is None:
                        continue
                    if ts.year < 2004 or ts.year > 2019:
                        continue

                    # Bank: try to match from item name, then from ITEM_NAME1..3 fields
                    resolved_bank = bank
                    if not resolved_bank:
                        for suffix in ("1", "2", "3"):
                            candidate = row.get(f"ITEM_NAME{suffix}", "")
                            resolved_bank = ecos_item_to_bank(candidate)
                            if resolved_bank:
                                break

                    # If still no bank, it may be an industry average → store with bank="ALL_BANKS_AVG"
                    bank_name = resolved_bank or "ALL_BANKS_AVG"
                    bank_group = "aggregate" if bank_name == "ALL_BANKS_AVG" else next(
                        (b["bank_group"] for b in BANK_SOURCE_REGISTRY if b["bank"] == bank_name),
                        "unknown",
                    )

                    record = {
                        "bank": bank_name,
                        "bank_group": bank_group,
                        "date": ts.strftime("%Y-%m-%d"),
                        "year": int(ts.year),
                        "quarter": f"Q{ts.quarter}",
                        "frequency": freq_label,
                        "currency": currency,
                        "product_type": product_type,
                        "maturity": maturity,
                        "resident_flag": "unknown",
                        "rate_percent": rate,
                        "source_url": source_url,
                        "source_type": "ecos",
                        "source_original_url": source_url,
                        "snapshot_timestamp": None,
                        "page_title": f"ECOS {stat_code}: {item_name}",
                        "raw_row_text": json.dumps(row, ensure_ascii=False),
                        "raw_column_text": item_name,
                    }
                    all_records.append(record)

                # If we got data for this cycle, don't try other cycles
                break

    log_event("info", "ECOS", "collect", f"Total ECOS records: {len(all_records)}", source_type="ecos")
    return all_records


# ── Current-page discovery ───────────────────────────────────────────────────

def discover_current_candidates(bank_meta: dict) -> list[str]:
    candidates: list[str] = list(bank_meta.get("known_rate_paths", []))
    for landing_url in bank_meta["landing_urls"]:
        resp = get_response_text(landing_url, bank_meta["bank"], "current_discovery", "current")
        if not resp["ok"] or "html" not in resp["content_type"].lower():
            continue
        parsed = parse_html_bundle(resp["text"], resp["final_url"])
        candidates.append(landing_url)
        scored: list[tuple[int, str]] = []
        for link in parsed["links"]:
            orig = strip_wayback_prefix(link["url"])
            if not same_bank_domain(orig, bank_meta["domains"]):
                continue
            s = score_candidate_link(link["text"], orig)
            if s > 0:
                scored.append((s, orig))
        for _, url in sorted(scored, key=lambda x: (-x[0], x[1]))[
            : RUN_CONFIG["max_current_links"]
        ]:
            candidates.append(url)
    deduped = list(dict.fromkeys(candidates))
    log_event(
        "info", bank_meta["bank"], "current_discovery",
        f"Discovered {len(deduped)} current candidates",
    )
    return deduped


# ── Wayback Machine: CDX API + availability API ─────────────────────────────

def cdx_search_snapshots(
    original_url: str, start_year: int = 2004, end_year: int = 2019
) -> list[dict]:
    """Use the CDX Server API for fine-grained snapshot discovery."""
    cdx_url = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": original_url,
        "output": "json",
        "fl": "timestamp,original,statuscode,mimetype",
        "filter": "statuscode:200",
        "filter": "mimetype:text/html",
        "from": f"{start_year}0101",
        "to": f"{end_year}1231",
        "limit": RUN_CONFIG["max_cdx_snapshots"],
        "collapse": "timestamp:6",  # one per month
    }
    try:
        resp = session.get(cdx_url, timeout=RUN_CONFIG["request_timeout"], params=params)
        if not resp.ok:
            return []
        rows = resp.json()
        if len(rows) < 2:
            return []
        header = rows[0]
        return [dict(zip(header, r)) for r in rows[1:]]
    except Exception as exc:
        log_event("error", "WAYBACK", "cdx_search", str(exc), source_url=original_url, source_type="wayback")
        return []


def lookup_wayback_snapshot(
    original_url: str, target_date: pd.Timestamp, bank: str
) -> dict | None:
    cache_key = (original_url, target_date.strftime("%Y-%m-%d"))
    if cache_key in WAYBACK_CACHE:
        return WAYBACK_CACHE[cache_key]
    lookup_url = (
        "https://archive.org/wayback/available?url="
        + quote(original_url, safe="/:?=&")
        + "&timestamp="
        + target_date.strftime("%Y%m%d000000")
    )
    try:
        payload = session.get(lookup_url, timeout=RUN_CONFIG["request_timeout"]).json()
        snapshot = payload.get("archived_snapshots", {}).get("closest")
        if snapshot and snapshot.get("available"):
            result = {
                "original_url": original_url,
                "archive_url": snapshot["url"],
                "snapshot_timestamp": snapshot["timestamp"],
                "status": snapshot.get("status"),
            }
            log_event(
                "info", bank, "wayback_lookup", "Snapshot found",
                source_url=snapshot["url"], source_type="wayback",
                target_date=target_date.strftime("%Y-%m-%d"),
            )
        else:
            result = None
            log_event(
                "warning", bank, "wayback_lookup", "No snapshot found",
                source_url=original_url, source_type="wayback",
                target_date=target_date.strftime("%Y-%m-%d"),
            )
    except Exception as exc:
        result = None
        log_event(
            "error", bank, "wayback_lookup", str(exc),
            source_url=original_url, source_type="wayback",
            target_date=target_date.strftime("%Y-%m-%d"),
        )
    WAYBACK_CACHE[cache_key] = result
    time.sleep(RUN_CONFIG["pause_seconds"])
    return result


def discover_wayback_candidates(
    bank_meta: dict, target_date: pd.Timestamp
) -> list[str]:
    candidates: list[str] = []

    # 1) Use known rate page paths via CDX for targeted discovery
    for rate_url in bank_meta.get("known_rate_paths", []):
        snapshot = lookup_wayback_snapshot(rate_url, target_date, bank_meta["bank"])
        if snapshot:
            candidates.append(rate_url)

    # 2) Use landing URLs as fallback
    for landing_url in bank_meta["landing_urls"]:
        snapshot = lookup_wayback_snapshot(landing_url, target_date, bank_meta["bank"])
        if not snapshot:
            continue
        archived = get_response_text(
            snapshot["archive_url"], bank_meta["bank"], "wayback_landing", "wayback"
        )
        if not archived["ok"] or "html" not in archived["content_type"].lower():
            continue
        parsed = parse_html_bundle(archived["text"], archived["final_url"])
        scored: list[tuple[int, str]] = []
        for link in parsed["links"]:
            orig = strip_wayback_prefix(link["url"])
            if not same_bank_domain(orig, bank_meta["domains"]):
                continue
            s = score_candidate_link(link["text"], orig)
            if s > 0:
                scored.append((s, orig))
        for _, url in sorted(scored, key=lambda x: (-x[0], x[1]))[
            : RUN_CONFIG["max_wayback_pages_per_date"]
        ]:
            candidates.append(url)

    return list(dict.fromkeys(candidates))


def fetch_wayback_page(
    original_url: str, target_date: pd.Timestamp, bank: str
) -> dict | None:
    snapshot = lookup_wayback_snapshot(original_url, target_date, bank)
    if not snapshot:
        return None
    archived = get_response_text(
        snapshot["archive_url"], bank, "wayback_fetch", "wayback"
    )
    if not archived["ok"] or "html" not in archived["content_type"].lower():
        return None
    return {
        "original_url": original_url,
        "archive_url": snapshot["archive_url"],
        "snapshot_timestamp": snapshot["snapshot_timestamp"],
        "html": archived["text"],
        "final_url": archived["final_url"],
    }


reset_logging_state()
print(f"Target dates prepared: {len(TARGET_DATES)}")
print(f"ECOS API client ready: {ECOS_CONFIG['base_url']}")

## 4. Table Parsing and Standardization

- **Label matching** — canonical currency, product, maturity, and resident flags via regex
- **Robust table extraction** — handles multi-level columns, mixed Korean/English labels
- **Numeric cleaning** — extracts rates in (0, 40] range, rejects non-numeric noise
- **Date attachment** — adds `year`, `quarter`, `frequency` fields from target date

In [ ]:
def match_canonical(
    text: str, mapping: dict[str, list[str]], default: str | None = None
) -> str | None:
    sample = normalize_space(text).lower()
    for canonical, patterns in mapping.items():
        if any(re.search(pat, sample, flags=re.IGNORECASE) for pat in patterns):
            return canonical
    return default


def canonical_currency(text: str) -> str | None:
    return match_canonical(text, CURRENCY_PATTERNS)


def canonical_product_type(text: str) -> str:
    return match_canonical(text, PRODUCT_PATTERNS, default="unknown")


def canonical_maturity(text: str) -> str:
    return match_canonical(text, MATURITY_PATTERNS, default="unknown")


def canonical_resident_flag(text: str) -> str:
    return match_canonical(text, RESIDENT_PATTERNS, default="unknown")


def flatten_columns(df: pd.DataFrame) -> list[str]:
    if isinstance(df.columns, pd.MultiIndex):
        return [
            normalize_space(" ".join(str(p) for p in col if str(p) != "nan"))
            for col in df.columns
        ]
    return [normalize_space(c) for c in df.columns]


def extract_numeric_rate(value: object) -> float | None:
    if pd.isna(value):
        return None
    sample = normalize_space(value)
    if not sample or sample == "-":
        return None
    sample = sample.replace(",", "").replace("%", "")
    matches = re.findall(r"-?\d+(?:\.\d+)?", sample)
    if not matches:
        return None
    number = float(matches[0])
    if number <= 0 or number > 40:
        return None
    return number


def attach_date_fields(target_date: pd.Timestamp) -> dict:
    target_date = pd.Timestamp(target_date)
    return {
        "date": target_date.strftime("%Y-%m-%d"),
        "year": int(target_date.year),
        "quarter": f"Q{target_date.quarter}",
        "frequency": FREQUENCY_LABEL,
    }


def extract_records_from_table(
    df: pd.DataFrame,
    bank_meta: dict,
    target_date: pd.Timestamp,
    source_url: str,
    source_type: str,
    source_original_url: str,
    page_title: str = "",
    snapshot_timestamp: str | None = None,
) -> list[dict]:
    df = df.copy()
    df.columns = flatten_columns(df)
    records: list[dict] = []
    for _, row in df.iterrows():
        row_map = {str(c): row[c] for c in df.columns}
        row_text = normalize_space(" ".join(str(v) for v in row_map.values()))
        row_currency = canonical_currency(row_text)
        row_product = canonical_product_type(row_text + " " + page_title)
        row_maturity = canonical_maturity(row_text + " " + page_title)
        row_resident = canonical_resident_flag(row_text)
        for col_name, raw_value in row_map.items():
            col_text = normalize_space(col_name)
            currency = canonical_currency(col_text) or row_currency
            if not currency:
                continue
            rate = extract_numeric_rate(raw_value)
            if rate is None:
                continue
            product_type = (
                row_product
                if row_product != "unknown"
                else canonical_product_type(col_text + " " + page_title)
            )
            maturity = (
                row_maturity
                if row_maturity != "unknown"
                else canonical_maturity(col_text + " " + page_title)
            )
            resident_flag = (
                row_resident
                if row_resident != "unknown"
                else canonical_resident_flag(col_text)
            )
            record = {
                "bank": bank_meta["bank"],
                "bank_group": bank_meta["bank_group"],
                **attach_date_fields(target_date),
                "currency": currency,
                "product_type": product_type,
                "maturity": maturity,
                "resident_flag": resident_flag,
                "rate_percent": float(rate),
                "source_url": source_url,
                "source_type": source_type,
                "source_original_url": source_original_url,
                "snapshot_timestamp": snapshot_timestamp,
                "page_title": page_title,
                "raw_row_text": row_text,
                "raw_column_text": col_text,
            }
            records.append(record)
    return records


def extract_records_from_page(
    parsed_page: dict,
    bank_meta: dict,
    target_date: pd.Timestamp,
    source_url: str,
    source_type: str,
    source_original_url: str,
    snapshot_timestamp: str | None = None,
) -> list[dict]:
    page_title = parsed_page.get("title", "")
    records: list[dict] = []
    tables = parsed_page.get("tables", [])[: RUN_CONFIG["max_tables_per_page"]]
    if not tables:
        log_event(
            "warning", bank_meta["bank"], "table_parse",
            "No HTML tables found",
            source_url=source_url, source_type=source_type,
            target_date=target_date.strftime("%Y-%m-%d"),
        )
        return records
    for table in tables:
        try:
            table_records = extract_records_from_table(
                table,
                bank_meta=bank_meta,
                target_date=target_date,
                source_url=source_url,
                source_type=source_type,
                source_original_url=source_original_url,
                page_title=page_title,
                snapshot_timestamp=snapshot_timestamp,
            )
            records.extend(table_records)
        except Exception as exc:
            log_event(
                "error", bank_meta["bank"], "table_parse", str(exc),
                source_url=source_url, source_type=source_type,
                target_date=target_date.strftime("%Y-%m-%d"),
            )
    return records


print("Parsing and standardization helpers ready.")

## 5. Collection Orchestration: ECOS → Web Scraping → Manual → Panel Construction

**Collection order:**
1. ECOS API (highest-quality official data)
2. Current official bank pages
3. Wayback Machine archived pages
4. Manual official disclosures

**Deduplication:** `manual > ecos > fss > current > wayback`, then by `term deposit > ordinary`, then `12M > 6M > 3M`

**Panel construction:**
- Extended panel = all standardized observations
- Strict panel = one product+maturity specification per bank for maximum comparability

In [ ]:
def ensure_schema(df: pd.DataFrame) -> pd.DataFrame:
    for col in REQUIRED_COLUMNS + EXTRA_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    ordered = REQUIRED_COLUMNS + EXTRA_COLUMNS
    remainder = [c for c in df.columns if c not in ordered]
    return df[ordered + remainder]


def load_manual_tables(file_path: Path) -> list[pd.DataFrame]:
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return [pd.read_csv(file_path)]
    if suffix in {".xlsx", ".xls"}:
        wb = pd.ExcelFile(file_path)
        return [wb.parse(s) for s in wb.sheet_names]
    if suffix in {".html", ".htm"}:
        return pd.read_html(file_path)
    raise ValueError(f"Unsupported manual source format: {file_path.name}")


def load_manual_official_sources() -> list[dict]:
    manifest = pd.read_csv(MANIFEST_FILE)
    if manifest.empty:
        log_event("info", "system", "manual_sources", "No manual official source files listed")
        return []
    manual_records: list[dict] = []
    for row in manifest.to_dict("records"):
        fpath = MANUAL_DIR / str(row["file_name"])
        if not fpath.exists():
            log_event(
                "warning", row["bank"], "manual_sources",
                "Listed manual file not found",
                source_url=row.get("source_url"), source_type="manual",
            )
            continue
        bank_meta = next(
            (b for b in BANK_SOURCE_REGISTRY if b["bank"] == row["bank"]), None
        )
        if bank_meta is None:
            log_event(
                "warning", row["bank"], "manual_sources",
                "Bank not found in registry",
                source_url=row.get("source_url"), source_type="manual",
            )
            continue
        source_date = pd.Timestamp(row["source_date"])
        try:
            tables = load_manual_tables(fpath)
            for table in tables:
                manual_records.extend(
                    extract_records_from_table(
                        table,
                        bank_meta=bank_meta,
                        target_date=source_date,
                        source_url=row["source_url"],
                        source_type="manual",
                        source_original_url=row["source_url"],
                        page_title=fpath.name,
                    )
                )
        except Exception as exc:
            log_event(
                "error", row["bank"], "manual_sources", str(exc),
                source_url=row.get("source_url"), source_type="manual",
            )
    return manual_records


def collect_current_records(bank_meta: dict) -> list[dict]:
    records: list[dict] = []
    scrape_date = pd.Timestamp.utcnow().normalize()
    for candidate_url in discover_current_candidates(bank_meta)[
        : RUN_CONFIG["max_current_links"]
    ]:
        resp = get_response_text(candidate_url, bank_meta["bank"], "current_fetch", "current")
        if not resp["ok"] or "html" not in resp["content_type"].lower():
            continue
        parsed = parse_html_bundle(resp["text"], resp["final_url"])
        page_records = extract_records_from_page(
            parsed,
            bank_meta=bank_meta,
            target_date=scrape_date,
            source_url=resp["final_url"],
            source_type="current",
            source_original_url=strip_wayback_prefix(candidate_url),
        )
        records.extend(page_records)
    log_event(
        "info", bank_meta["bank"], "current_fetch",
        f"Current records collected: {len(records)}",
    )
    return records


def collect_wayback_records(bank_meta: dict) -> list[dict]:
    records: list[dict] = []
    for target_date in TARGET_DATES:
        candidate_urls = discover_wayback_candidates(bank_meta, target_date)
        for orig_url in candidate_urls[: RUN_CONFIG["max_wayback_pages_per_date"]]:
            page = fetch_wayback_page(orig_url, target_date, bank_meta["bank"])
            if not page:
                continue
            parsed = parse_html_bundle(page["html"], page["final_url"])
            page_records = extract_records_from_page(
                parsed,
                bank_meta=bank_meta,
                target_date=target_date,
                source_url=page["archive_url"],
                source_type="wayback",
                source_original_url=page["original_url"],
                snapshot_timestamp=page["snapshot_timestamp"],
            )
            records.extend(page_records)
    log_event(
        "info", bank_meta["bank"], "wayback_fetch",
        f"Wayback records collected: {len(records)}",
    )
    return records


def build_raw_dataset() -> pd.DataFrame:
    """
    Multi-stage collection: ECOS → Current pages → Wayback → Manual.
    All records are combined into the required schema.
    """
    collected: list[dict] = []

    # ── Stage 1: ECOS (Bank of Korea) ────────────────────────────────────
    print("Stage 1/4: Collecting from ECOS API...")
    ecos_records = ecos_collect_fx_deposit_rates()
    collected.extend(ecos_records)
    print(f"  → ECOS records: {len(ecos_records)}")

    # ── Stage 2: Current official bank pages ─────────────────────────────
    print("Stage 2/4: Scraping current bank pages...")
    for bank_meta in BANK_SOURCE_REGISTRY:
        current_recs = collect_current_records(bank_meta)
        collected.extend(current_recs)
        if current_recs:
            print(f"  → {bank_meta['bank']}: {len(current_recs)} current records")

    # ── Stage 3: Wayback Machine ─────────────────────────────────────────
    print("Stage 3/4: Fetching Wayback Machine snapshots...")
    for bank_meta in BANK_SOURCE_REGISTRY:
        wb_recs = collect_wayback_records(bank_meta)
        collected.extend(wb_recs)
        if wb_recs:
            print(f"  → {bank_meta['bank']}: {len(wb_recs)} wayback records")

    # ── Stage 4: Manual official disclosures ─────────────────────────────
    print("Stage 4/4: Loading manual official sources...")
    manual_recs = load_manual_official_sources()
    collected.extend(manual_recs)
    print(f"  → Manual records: {len(manual_recs)}")

    raw = pd.DataFrame(collected)
    raw = ensure_schema(raw)
    if raw.empty:
        print("\n⚠ No records collected from any source.")
        return raw

    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    raw = raw.dropna(subset=["date", "currency", "rate_percent", "source_url", "source_type"]).copy()
    raw["date"] = raw["date"].dt.strftime("%Y-%m-%d")
    raw["year"] = pd.to_datetime(raw["date"]).dt.year
    raw["quarter"] = "Q" + pd.to_datetime(raw["date"]).dt.quarter.astype(str)

    # Filter to 2004–2019 and exclude aggregate rows from the final bank-level dataset
    raw = raw[(raw["year"] >= 2004) & (raw["year"] <= 2019)].copy()

    print(f"\nRaw dataset: {len(raw)} rows, {raw['bank'].nunique()} distinct banks")
    print(f"  Banks: {sorted(raw['bank'].unique())}")
    print(f"  Currencies: {sorted(raw['currency'].unique())}")
    print(f"  Source types: {dict(raw['source_type'].value_counts())}")
    return raw.sort_values(
        ["bank", "date", "currency", "source_type", "rate_percent"]
    ).reset_index(drop=True)


def build_extended_panel(raw_df: pd.DataFrame) -> pd.DataFrame:
    if raw_df.empty:
        return ensure_schema(raw_df.copy())
    extended = raw_df.copy()
    extended["resident_flag"] = extended["resident_flag"].fillna("unknown")
    extended["product_type"] = extended["product_type"].fillna("unknown")
    extended["maturity"] = extended["maturity"].fillna("unknown")
    extended = extended.drop_duplicates().reset_index(drop=True)
    return ensure_schema(extended)


def deduplicate_observations(
    extended_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if extended_df.empty:
        return ensure_schema(extended_df.copy()), ensure_schema(extended_df.copy())
    ranked = extended_df.copy()
    ranked["source_rank"] = ranked["source_type"].map(SOURCE_PRIORITY).fillna(99)
    ranked["product_rank"] = ranked["product_type"].map(PRODUCT_PRIORITY).fillna(99)
    ranked["maturity_rank"] = ranked["maturity"].map(MATURITY_PRIORITY).fillna(99)
    ranked = ranked.sort_values(
        [
            "bank", "date", "currency", "resident_flag",
            "source_rank", "product_rank", "maturity_rank", "source_url",
        ]
    ).reset_index(drop=True)
    dup_keys = ["bank", "date", "currency", "resident_flag"]
    keep_mask = ~ranked.duplicated(subset=dup_keys, keep="first")
    clean_df = (
        ranked.loc[keep_mask]
        .drop(columns=["source_rank", "product_rank", "maturity_rank"])
        .reset_index(drop=True)
    )
    dropped_df = (
        ranked.loc[~keep_mask]
        .drop(columns=["source_rank", "product_rank", "maturity_rank"])
        .reset_index(drop=True)
    )
    return ensure_schema(clean_df), ensure_schema(dropped_df)


def choose_strict_panel_specs(extended_df: pd.DataFrame) -> pd.DataFrame:
    if extended_df.empty:
        return pd.DataFrame(
            columns=[
                "bank", "product_type", "maturity", "resident_flag",
                "score", "required_currency_count", "observation_count",
            ]
        )
    base = extended_df.copy()
    base["is_req"] = base["currency"].isin(RUN_CONFIG["required_currencies"]).astype(int)
    grouped = (
        base.groupby(
            ["bank", "product_type", "maturity", "resident_flag"], dropna=False
        )
        .agg(
            required_currency_count=(
                "currency",
                lambda v: len(
                    set(x for x in v if x in RUN_CONFIG["required_currencies"])
                ),
            ),
            observation_count=("date", "count"),
            start_date=("date", "min"),
            end_date=("date", "max"),
        )
        .reset_index()
    )
    grouped["product_rank"] = grouped["product_type"].map(PRODUCT_PRIORITY).fillna(99)
    grouped["maturity_rank"] = grouped["maturity"].map(MATURITY_PRIORITY).fillna(99)
    grouped["score"] = grouped["required_currency_count"] * 1000 + grouped["observation_count"]
    grouped = grouped.sort_values(
        ["bank", "score", "product_rank", "maturity_rank", "resident_flag"],
        ascending=[True, False, True, True, True],
    )
    return grouped.groupby("bank", as_index=False).head(1).reset_index(drop=True)


def build_strict_panel(
    extended_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    specs = choose_strict_panel_specs(extended_df)
    if specs.empty:
        return ensure_schema(extended_df.copy()), specs
    strict_panel = extended_df.merge(
        specs[["bank", "product_type", "maturity", "resident_flag"]],
        on=["bank", "product_type", "maturity", "resident_flag"],
        how="inner",
    )
    strict_panel = strict_panel.sort_values(
        ["bank", "date", "currency"]
    ).reset_index(drop=True)
    return ensure_schema(strict_panel), specs


print("Collection orchestration, deduplication, and panel selection ready.")

## 6. Coverage Reports, Quality Validation, and Visualizations

- **Missing report** — expected vs. actual observations per (bank, currency)
- **Quality flags** — identical-pattern, constant-series, too-smooth, impossible-range, missing-traceability
- **Line plots** — per-currency USD/JPY/CNY deposit rate time series
- **Bank comparison** — strict-panel overlay by currency

In [ ]:
def expected_observation_count() -> int:
    return len(TARGET_DATES)


def build_missing_report(panel_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict] = []
    expected = expected_observation_count()
    currencies = RUN_CONFIG["required_currencies"] + RUN_CONFIG["optional_currencies"]
    for bank_meta in BANK_SOURCE_REGISTRY:
        for currency in currencies:
            actual = 0
            if not panel_df.empty:
                actual = panel_df[
                    (panel_df["bank"] == bank_meta["bank"])
                    & (panel_df["currency"] == currency)
                ]["date"].nunique()
            rows.append(
                {
                    "bank": bank_meta["bank"],
                    "currency": currency,
                    "expected_observations": expected,
                    "actual_observations": int(actual),
                    "coverage_ratio": round(actual / expected, 4)
                    if expected
                    else np.nan,
                }
            )
    return pd.DataFrame(rows)


def quality_flags_from_dataframe(clean_df: pd.DataFrame) -> pd.DataFrame:
    flags: list[dict] = []
    if clean_df.empty:
        return pd.DataFrame(columns=["flag_type", "currency", "bank", "details"])

    # Impossible range
    bad = clean_df[(clean_df["rate_percent"] <= 0) | (clean_df["rate_percent"] > 40)]
    for _, r in bad.iterrows():
        flags.append(
            {
                "flag_type": "impossible_range",
                "currency": r["currency"],
                "bank": r["bank"],
                "details": f"rate_percent={r['rate_percent']} on {r['date']}",
            }
        )

    # Missing traceability
    missing_src = clean_df[
        clean_df[["source_url", "source_type"]].isna().any(axis=1)
    ]
    for _, r in missing_src.iterrows():
        flags.append(
            {
                "flag_type": "missing_traceability",
                "currency": r["currency"],
                "bank": r["bank"],
                "details": f"missing source fields on {r['date']}",
            }
        )

    # Duplicate source reuse
    dup_src = (
        clean_df.groupby(["bank", "date", "currency", "source_url"])
        .size()
        .reset_index(name="count")
    )
    dup_src = dup_src[dup_src["count"] > 1]
    for _, r in dup_src.iterrows():
        flags.append(
            {
                "flag_type": "duplicate_source_url",
                "currency": r["currency"],
                "bank": r["bank"],
                "details": f"{r['source_url']} reused {r['count']} times on {r['date']}",
            }
        )

    # Constant series and too-smooth detection
    for (bank, currency), subset in clean_df.groupby(["bank", "currency"]):
        vals = subset.sort_values("date")["rate_percent"].dropna().tolist()
        if len(vals) > 1 and len(set(vals)) == 1:
            flags.append(
                {
                    "flag_type": "constant_series",
                    "currency": currency,
                    "bank": bank,
                    "details": f"{len(vals)} repeated values",
                }
            )
        if len(vals) >= 4:
            diffs = np.diff(vals)
            if np.nanstd(diffs) < 0.05:
                flags.append(
                    {
                        "flag_type": "too_smooth",
                        "currency": currency,
                        "bank": bank,
                        "details": f"std(diff)={np.nanstd(diffs):.4f}",
                    }
                )

    # Identical patterns across banks
    for currency, subset in clean_df.groupby("currency"):
        pivot = subset.pivot_table(
            index="date", columns="bank", values="rate_percent", aggfunc="first"
        )
        for left, right in combinations(pivot.columns, 2):
            paired = pivot[[left, right]].dropna()
            if len(paired) >= 3 and paired[left].equals(paired[right]):
                flags.append(
                    {
                        "flag_type": "identical_pattern",
                        "currency": currency,
                        "bank": f"{left} vs {right}",
                        "details": f"identical values across {len(paired)} observations",
                    }
                )

    flags_df = pd.DataFrame(flags)
    for r in flags_df.to_dict("records"):
        log_event(
            "warning",
            r["bank"],
            "quality_check",
            r["details"],
            source_type="quality_check",
            extra={"flag_type": r["flag_type"], "currency": r["currency"]},
        )
    return flags_df


# ── Visualizations ───────────────────────────────────────────────────────────

def plot_currency_lines(
    panel_df: pd.DataFrame, currency: str, file_name: str
) -> None:
    subset = panel_df[panel_df["currency"] == currency].copy()
    if subset.empty:
        log_event("info", "system", "plotting", f"No data for {currency} plot")
        return
    subset["date"] = pd.to_datetime(subset["date"])
    fig, ax = plt.subplots(figsize=(10, 5))
    for bank, series in subset.groupby("bank"):
        series = series.sort_values("date")
        ax.plot(
            series["date"], series["rate_percent"],
            marker="o", linewidth=1.6, label=bank,
        )
    ax.set_title(f"{currency} Foreign Currency Deposit Rates (2004–2019)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rate (%)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / file_name, bbox_inches="tight", dpi=140)
    plt.close(fig)
    print(f"  Saved: {FIG_DIR / file_name}")


def plot_bank_comparison(strict_panel: pd.DataFrame, file_name: str) -> None:
    required = RUN_CONFIG["required_currencies"]
    fig, axes = plt.subplots(
        len(required), 1, figsize=(11, 4 * len(required)), sharex=True
    )
    if len(required) == 1:
        axes = [axes]
    for axis, currency in zip(axes, required):
        subset = strict_panel[strict_panel["currency"] == currency].copy()
        subset["date"] = pd.to_datetime(subset["date"])
        for bank, series in subset.groupby("bank"):
            series = series.sort_values("date")
            axis.plot(
                series["date"], series["rate_percent"],
                marker="o", linewidth=1.5, label=bank,
            )
        axis.set_title(f"Strict-Panel Comparison: {currency}")
        axis.set_ylabel("Rate (%)")
        axis.grid(True, alpha=0.25)
        axis.legend(loc="best", fontsize=8)
    axes[-1].set_xlabel("Date")
    fig.tight_layout()
    fig.savefig(FIG_DIR / file_name, bbox_inches="tight", dpi=140)
    plt.close(fig)
    print(f"  Saved: {FIG_DIR / file_name}")


print("Reporting, validation, and plotting helpers ready.")

## 7. Export and Run Pipeline

**CSV outputs:** `fx_deposit_rates_raw.csv`, `fx_deposit_rates_clean.csv`

**Excel output:** `fx_deposit_rates_panel.xlsx` with 6 sheets:
- `raw` — all collected observations
- `clean` — deduplicated with priority ranking
- `strict_panel` — one product/maturity per bank for comparability
- `extended_panel` — all standardized observations with metadata
- `source_log` — every request, parse event, quality warning
- `missing_report` — coverage gaps per (bank, currency)

In [ ]:
def export_deliverables(
    raw_df: pd.DataFrame,
    clean_df: pd.DataFrame,
    strict_panel: pd.DataFrame,
    extended_panel: pd.DataFrame,
    missing_report: pd.DataFrame,
    dropped_df: pd.DataFrame,
) -> pd.DataFrame:
    source_log = pd.DataFrame(SOURCE_EVENTS)

    raw_df = ensure_schema(raw_df)
    clean_df = ensure_schema(clean_df)
    strict_panel = ensure_schema(strict_panel)
    extended_panel = ensure_schema(extended_panel)

    # CSV outputs
    raw_df.to_csv(OUTPUT_RAW_CSV, index=False, encoding="utf-8-sig")
    clean_df.to_csv(OUTPUT_CLEAN_CSV, index=False, encoding="utf-8-sig")
    dropped_df.to_csv(OUTPUT_DROPPED_CSV, index=False, encoding="utf-8-sig")
    source_log.to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

    # Excel output with formatted sheets
    with pd.ExcelWriter(OUTPUT_PANEL_XLSX, engine="xlsxwriter") as writer:
        sheet_map = {
            "raw": raw_df,
            "clean": clean_df,
            "strict_panel": strict_panel,
            "extended_panel": extended_panel,
            "source_log": source_log,
            "missing_report": missing_report,
        }
        for sheet_name, frame in sheet_map.items():
            frame.to_excel(writer, sheet_name=sheet_name, index=False)

        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#D9E2F3"})
        for sheet_name, frame in sheet_map.items():
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 0)
            ws.set_row(0, None, header_fmt)
            ncols = max(len(frame.columns) - 1, 0)
            ws.autofilter(0, 0, max(len(frame), 1), ncols)
            ws.set_column(0, ncols, 18)

    return source_log


def run_pipeline() -> dict:
    """Execute the full multi-stage pipeline end-to-end."""
    reset_logging_state()
    print("=" * 70)
    print("FX Deposit Rate Pipeline — START")
    print("=" * 70)

    # Collect
    raw_df = build_raw_dataset()

    # Panels
    extended_panel = build_extended_panel(raw_df)
    clean_df, dropped_df = deduplicate_observations(extended_panel)
    strict_panel, strict_specs = build_strict_panel(extended_panel)
    missing_report = build_missing_report(extended_panel)

    # Quality
    quality_flags = quality_flags_from_dataframe(clean_df)

    # Plots
    print("\nGenerating plots...")
    plot_currency_lines(strict_panel, "USD", "fx_rates_usd.png")
    plot_currency_lines(strict_panel, "JPY", "fx_rates_jpy.png")
    plot_currency_lines(strict_panel, "CNY", "fx_rates_cny.png")
    plot_bank_comparison(strict_panel, "bank_comparison_strict_panel.png")

    # Export
    print("\nExporting deliverables...")
    source_log = export_deliverables(
        raw_df, clean_df, strict_panel, extended_panel, missing_report, dropped_df
    )

    print("\n" + "=" * 70)
    print("FX Deposit Rate Pipeline — COMPLETE")
    print("=" * 70)

    return {
        "raw": raw_df,
        "clean": clean_df,
        "strict_panel": strict_panel,
        "extended_panel": extended_panel,
        "strict_specs": strict_specs,
        "missing_report": missing_report,
        "quality_flags": quality_flags,
        "source_log": source_log,
        "dropped_duplicates": dropped_df,
    }


# ── Execute ──────────────────────────────────────────────────────────────────
results = run_pipeline()

# ── Summary ──────────────────────────────────────────────────────────────────
summary = pd.DataFrame(
    {
        "table": [
            "raw", "clean", "strict_panel", "extended_panel",
            "missing_report", "quality_flags", "source_log",
        ],
        "rows": [
            len(results["raw"]),
            len(results["clean"]),
            len(results["strict_panel"]),
            len(results["extended_panel"]),
            len(results["missing_report"]),
            len(results["quality_flags"]),
            len(results["source_log"]),
        ],
    }
)

print("\n── Pipeline Summary ──")
print(summary.to_string(index=False))
print(f"\nOutputs written:")
print(f"  {OUTPUT_RAW_CSV.name}")
print(f"  {OUTPUT_CLEAN_CSV.name}")
print(f"  {OUTPUT_PANEL_XLSX.name}")
print(f"  {OUTPUT_DROPPED_CSV.name}")
print(f"  figures/ → {FIG_DIR.name}/")

if not results["quality_flags"].empty:
    print(f"\n── Quality Flags ({len(results['quality_flags'])}) ──")
    print(results["quality_flags"].to_string(index=False))

if not results["missing_report"].empty:
    print(f"\n── Missing Report ──")
    print(results["missing_report"].to_string(index=False))

if not results["strict_specs"].empty:
    print(f"\n── Strict Panel Specs ──")
    print(results["strict_specs"][["bank", "product_type", "maturity", "score"]].to_string(index=False))

summary